Tarea 3 :

Respuesta  Pregunta 1:

DataFrame de 31 055 filas y 23 columnas.

Entre las variables: datos sociodemográficos (sexo, edad, vive_padre, vive_madre, area, educm, educp, madre_work), z‐score de IMCE (imce), 13 ítems socioemocionales (sk1–sk13) y nivel de actividad física (act_fisica).

Valores con nulos y limpiezas extras:
Solo act_fisica (1004 NaN, ≈3,2 %) y educm (373 NaN, ≈1,2 %) tenían NaNs.

Se comprobó que los nulos en act_fisica estaban distribuidos casi aleatoriamente por sexo y área (MCAR), y que en educm existía un ligero sesgo (MAR) hacia sexo = 1, pero el porcentaje afectado era muy bajo.

Se optó por eliminar mediante dropna las filas que tuvieran NaN en act_fisica o educm debido a que no afectan significativamente el estudio.

Se eliminaron duplicados.



In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../../data/junaeb3.csv', encoding='utf-8')  
print(f'Filas, columnas  ➜  {df.shape}')
display(df.head())         
df.info()                  
display(df.describe(include='all').T)
na_counts = df.isna().sum().sort_values(ascending=False)
print('\nNulos por variable (solo las que tienen):')
display(na_counts[na_counts > 0])
plt.figure(figsize=(10, 4))
na_counts[na_counts > 0].plot.bar()
plt.ylabel('Cantidad de NaN')
plt.title('Valores faltantes por variable')
plt.tight_layout()
plt.show()
LIKERT_TO_REVERSE = [
]
for col in LIKERT_TO_REVERSE:
    if col in df.columns:
        df[col] = df[col].map({1:5, 2:4, 3:3, 4:2, 5:1})
        print(f'Revertido sentido de {col}')
before = df.shape[0]
df = df.drop_duplicates()
print(f'Duplicados removidos: {before - df.shape[0]}')


In [ ]:
print(df[df['act_fisica'].isna()]['sexo'].value_counts(normalize=True))
print(df[df['act_fisica'].isna()]['area'].value_counts(normalize=True))

print(df[df['educm'].isna()]['sexo'].value_counts(normalize=True))
print(df[df['educm'].isna()]['area'].value_counts(normalize=True))


In [ ]:
df = df.dropna(subset=['act_fisica', 'educm']).copy()

Respuesta pregunta 2:

Para evaluar la fiabilidad de la escala socio-emocional se calcularon el alfa de Cronbach global. Tambien la correlación ítem-total y el alfa que resultaría al eliminarlo para cada item. 

El coeficiente global fue 0,775, valor que indica una consistencia interna adecuada. 

El ítem sk7 mostró una correlación ítem-total muy baja (0,14). Y al excluirlo el alfa aumenta a 0,807, lo cual mejora la homogeneidad de la escala.

Se comprobaron los supuestos de factorabilidad: el índice KMO alcanzó 0,872 y el test de Bartlett resultó altamente significativo (χ² = 88 989; p < 0,001), confirmando que las correlaciones son suficientes para un análisis factorial.

Para futuros test y modelos se decidió retirar sk7 y continuar el Análisis Factorial Exploratorio con los 12 ítems restantes (sk1–sk6 y sk8–sk13).

In [ ]:


import numpy as np
import pandas as pd
from factor_analyzer.factor_analyzer import (
    calculate_kmo,
    calculate_bartlett_sphericity,
)

items = [f"sk{i}" for i in range(1, 14)]
X = df[items]                  

k = X.shape[1]                 
item_var = X.var(axis=0, ddof=1).sum()
total_score = X.sum(axis=1)
total_var = total_score.var(ddof=1)
alpha_global = (k / (k - 1)) * (1 - item_var / total_var)
print(f"\nAlfa de Cronbach (global) = {alpha_global:.3f}")

alpha_if_deleted, item_total_corr = {}, {}
for col in items:
    X_drop = X.drop(columns=[col])
    k_sub = X_drop.shape[1]
    item_var_sub = X_drop.var(axis=0, ddof=1).sum()
    total_sub = X_drop.sum(axis=1)
    total_var_sub = total_sub.var(ddof=1)
    alpha_if_deleted[col] = (k_sub / (k_sub - 1)) * (1 - item_var_sub / total_var_sub)
    item_total_corr[col] = X[col].corr(total_score - X[col])

reliab = (
    pd.DataFrame({
        "item_total_corr": item_total_corr,
        "alpha_if_deleted": alpha_if_deleted,
    })
    .round(3)
    .sort_values("item_total_corr")
)
display(reliab)             

kmo_all, kmo_model = calculate_kmo(X)
chi2, pval = calculate_bartlett_sphericity(X)

print(f"\nKMO general   = {kmo_model:.3f}")
print(f"Test Bartlett = χ²({X.shape[1]*(X.shape[1]-1)//2}) = {chi2:.1f},  p = {pval:.3e}")


Respuesta pregunta 3:

Se aplicó un EFA con extracción de componentes principales y rotación Varimax sobre los 12 ítems restantes (se excluyó sk7).
(Hubo intento de usar y seguir el procedimiento del codigo guia en sem.ipynb)


El scree-plot muestra un codo claro después del segundo factor.

Slo los tres primeros eigen-valores superan la unidad (4,09 ; 1,34 ; 1,08).

Se retuvo una solución de dos factores, coherente con la teoría de conductas prosociales vs. autorregulación/emoción


Factor 1 agrupa con cargas altas los ítems sk1-sk6 (0,52 – 0,73) y, en menor medida, sk8 (0,38).
Se interpreta como “Competencia social / prosocial”.

Factor 2 concentra sk9-sk13 (0,60 – 0,71).
Se interpreta como “Regulación emocional / autocontrol”.

sk8 presenta una carga simétrica en ambos factores (0,38 en F1 y 0,38 en F2), por lo que se podría considerar un ítem de cruce.
Se mantendrá en F1 de momento.


Varianza explicada: 34 % (F1) + 11 % (F2) = 45 % total

KMO global ya funciona de buena manera con 0,87 y Bartlett significativo, asegurando que es un modelo adecuado (por el momento)

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from factor_analyzer import FactorAnalyzer, Rotator
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

items = [f"sk{i}" for i in range(1, 14) if i != 7]   
X = df[items]                                        

corr = X.corr()
eigen_vals, _ = np.linalg.eig(corr)
eigen_vals = np.sort(eigen_vals)[::-1]

eig_df = pd.DataFrame({
    "Factor": range(1, len(eigen_vals) + 1),
    "Eigenvalue": eigen_vals.round(3),
    "λ > 1": eigen_vals > 1
})
print("\nEigen-valores:")
display(eig_df)

plt.figure(figsize=(6,4))
plt.plot(range(1, len(eigen_vals)+1), eigen_vals, marker="o")
plt.axhline(1, ls="--", lw=1)
plt.title("Scree plot – escala SK (12 ítems)")
plt.xlabel("Factor")
plt.ylabel("Eigen-value")
plt.tight_layout()
plt.show()

n_factors = 2     

fa = FactorAnalyzer(n_factors=n_factors, rotation=None, method="principal")
fa.fit(X)

rotator = Rotator(method="varimax")
loadings_rot = pd.DataFrame(
    rotator.fit_transform(fa.loadings_),
    index=items,
    columns=[f"F{i+1}" for i in range(n_factors)]
).round(3)

communalities = pd.Series(fa.get_communalities(), index=items).round(3)
variance = pd.Series(fa.get_factor_variance()[1], index=[f"F{i+1}" for i in range(n_factors)]).round(3)

print("\nCargas factoriales rotadas (|carga| ≥ 0.30 en negrita):")
display(loadings_rot.style.format("{:.3f}").applymap(lambda v: "font-weight:bold" if abs(v)>=0.30 else ""))

print("\nComunalidades:")
display(communalities)

print("\nVarianza explicada por factor (%):")
display(variance * 100)


Respuesta pregunta 4:

Se ajustó un CFA con dos factores correlacionados.

se probó el modelo con los 12 ítems conservados en el EFA (sk1–sk6, sk8, sk9–sk13).

Dado que los índices quedaron algo bajos, se reestimó el modelo eliminando sk8.

Resultados de ajuste

Modelo 12 ítems: CFI = 0,894; TLI = 0,868; RMSEA = 0,074 . ajuste marginal.

Modelo 11 ítems (sin sk8): CFI = 0,926; TLI = 0,906; RMSEA = 0,065 . ajuste bueno (cumple CFI/TLI ≥ 0,90 y RMSEA ≤ 0,08).

Cargas y varianza explicada

Las cargas estandarizadas de los 11 ítems variaron entre 0,50 y 0,80, todas suficientes.

Los R2 no los devuelve semopy en un CFA puro, pero se calcularon como (carga2); varían de ~0,25 a ~0,65, indicando que cada indicador explica entre el 25 % y el 65 % de su propia varianza.



Se adopta el modelo definitivo de 11 ítems, con dos factores:
F1 (Competencia social): sk1–sk6.
F2 (Regulación emocional): sk9–sk13.

El ítem sk8 se descarta porque deterioraba el ajuste sin aportar información sustantiva (ya previsto desde la pregunta 2).

In [ ]:

from semopy import Model, calc_stats
import pandas as pd
import re

def run_cfa(model_syntax, item_list, data, title=""):
    print(f"\n=== {title} ===")
    mod = Model(model_syntax)
    mod.fit(data[item_list])

    stats = calc_stats(mod).squeeze()     
    stats.index = (
        stats.index
             .str.lower()
             .str.replace(r"[-_]", "", regex=True)
    )

    def g(key):
        return stats.get(key, float("nan"))

    print(f"χ²({g('dof'):.0f}) = {g('chi2'):.1f},  p = {g('pvalue'):.3g}")
    print(f"CFI={g('cfi'):.3f}  TLI={g('tli'):.3f}  "
          f"RMSEA={g('rmsea'):.3f}  SRMR={g('srmr'):.3f}\n")

    pt = mod.inspect(std_est=True)

    load = (
        pt[pt.op == '~'][['lval', 'rval', 'Estimate']]
          .rename(columns={'lval':'Item', 'rval':'Factor',
                           'Estimate':'Std_Loading'})
          .set_index('Item')
    )

    r2 = (
        pt[pt.op == 'R2'][['lval','Estimate']]
          .rename(columns={'lval':'Item','Estimate':'R2'})
          .set_index('Item')
    )

    res = load.join(r2).round(3).sort_values('Std_Loading')
    display(res)
    return g('cfi'), g('tli')

items_12 = [f"sk{i}" for i in range(1,14) if i != 7]    # con sk8
items_11 = [f"sk{i}" for i in range(1,14) if i not in (7,8)]

model_12 = """
F1 =~ sk1 + sk2 + sk3 + sk4 + sk5 + sk6 + sk8
F2 =~ sk9 + sk10 + sk11 + sk12 + sk13
F1 ~~ F2
"""

model_11 = """
F1 =~ sk1 + sk2 + sk3 + sk4 + sk5 + sk6
F2 =~ sk9 + sk10 + sk11 + sk12 + sk13
F1 ~~ F2
"""

cfi12, tli12 = run_cfa(model_12, items_12, df, "Modelo 12 ítems (con sk8)")

if (cfi12 < .90) or (tli12 < .90):
    run_cfa(model_11, items_11, df, "Modelo 11 ítems (sin sk8)")


Respuesta pregunta 5:

F1 (competencia social): sk1–sk6
F2 (regulación emocional) :sk9–sk13

Ajuste global del modelo:

χ2(148) = 10 981.5 (p < 0.001)

CFI = 0.889, TLI = 0.867 (marginal, < 0.90)

RMSEA = 0.050 (bueno, ≤ 0.06)

SRMR no devuelto por la versión de semopy. (lanzaba keyerror)


Cargas: todos los ítems cargan fuertemente en su factor (sk2–sk6 entre 1.36 y 1.95; sk10–sk13 entre 1.10 y 1.36).

Efectos sobre IMCE:

F1: IMCE = –0.02 (prácticamente nulo)

F2: IMCE = 0.02 (prácticamente nulo)

Sexo (0.13) y área (–0.12) son los predictores más claros: los niños (sexo = 1) tienden a mayor IMCE y quienes viven en zona urbana a un IMCE más bajo, manteniendo las demás variables constantes.

Efectos sobre actividad física

F1: AF = –0.17

F2: AF = –0.37
mayor puntaje socio-emocional, en especial en regulación emocional, se asocia con menor reporte de actividad física.

Sexo (0.15) y área (–0.27) también influyen: los niños son algo más activos y los urbanos menos activos.

Correlaciones

F1 -- F2 = 0.05 (casi independientes).

IMCE -- AF (residuos) ≈ –0.03 (muy baja, negativa).

In [ ]:

from semopy import Model, calc_stats
import pandas as pd
import re

def run_sem(model_syntax, item_list, outcomes, covars, data, title=""):
    print(f"\n=== {title} ===")
    sem = Model(model_syntax)
    sem.fit(data[item_list + outcomes + covars])
    
    stats = calc_stats(sem).squeeze()
    stats.index = stats.index.str.lower().str.replace(r"[-_]", "", regex=True)
    
    def G(key):
        return stats.get(key, float("nan"))
    
    print(f"χ²({G('dof'):.0f}) = {G('chi2'):.1f},  p = {G('pvalue'):.3g}")
    print(f"CFI={G('cfi'):.3f}  TLI={G('tli'):.3f}  RMSEA={G('rmsea'):.3f}  SRMR={G('srmr'):.3f}")
    
    pt = sem.inspect(std_est=True)
    params = (
        pt[pt.op.isin(['~', '~~'])]
          .loc[:, ['lval','op','rval','Estimate']]
          .rename(columns={'Estimate':'Std_Est'})
    )
    display(params.round(3))
    return stats

items = ['sk1','sk2','sk3','sk4','sk5','sk6',
         'sk9','sk10','sk11','sk12','sk13']
outcomes = ['imce', 'act_fisica']
covars = ['sexo', 'edad', 'area', 'educm', 'educp', 'madre_work']

sem_model = f"""
F1 =~ sk1 + sk2 + sk3 + sk4 + sk5 + sk6
F2 =~ sk9 + sk10 + sk11 + sk12 + sk13
F1 ~~ F2

imce       ~ F1 + F2 + {' + '.join(covars)}
act_fisica ~ F1 + F2 + {' + '.join(covars)}
imce ~~ act_fisica
"""
stats_sem = run_sem(sem_model, items, outcomes, covars, df, "SEM completo")


Respuesta pregunta 6:

Se ajustaron modelos de mezcla gaussiana con 1 ≤ k ≤ 5 componentes usando sólo el z-IMCE.

El criterio BIC se calculó para cada k y se graficó.


El BIC alcanzó su mínimo en k = 1.

Se concluye que un único componente gaussiano describe mejor la distribución del z-IMCE. No se detectan sub-poblaciones diferenciables según este indicador.

Descriptivos de la clase única:

Total de casos: 29 683 (100 %).

Media de IMCE: 1.018 (z-score).


Al existir una sola clase latente, no tiene sentido estimar el SEM por sub-muestras.

Si se quisiera segmentar, habría que incluir más variables (p. ej. perímetro de cintura, talla para la edad, o covariables dietéticas) o probar un rango de k mayor, pero con el IMCE aislado la población parece homogénea.

 no se identifican clases latentes distintas basadas sólo en el z-IMCE.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from matplotlib import pyplot as plt

X_imce = df[['imce']].values   

bics = []
models = {}
for k in range(1, 6):
    gm = GaussianMixture(n_components=k, covariance_type='full', random_state=0)
    gm.fit(X_imce)
    bic = gm.bic(X_imce)
    bics.append(bic)
    models[k] = gm

plt.figure(figsize=(5,3))
plt.plot(range(1, 6), bics, marker='o')
plt.xlabel("Número de clases (k)")
plt.ylabel("BIC")
plt.title("Selección de k por BIC")
plt.tight_layout()
plt.show()

k_opt = int(np.argmin(bics) + 1)
print(f"Modelo óptimo según BIC: k = {k_opt}")

gmm_opt = models[k_opt]
df['latent_class'] = gmm_opt.predict(X_imce)

class_counts = df['latent_class'].value_counts().sort_index()
class_means  = df.groupby('latent_class')['imce'].mean().round(3)
print("\nTamaño de cada clase:")
print(class_counts)
print("\nMedia de IMCE en cada clase:")
print(class_means)


Respuesta pregunta 7:

El análisis de mezcla (IMCE como única variable) arrojó k = 1 como solución óptima: toda la muestra pertenece a una sola clase latente.

no existen sub-poblaciones diferenciadas que justifiquen un multi-group SEM


No se encuentra relevancia al estimar el modelo por grupos cuando la LCA identifica una sola clase. cualquier comparación inter-clases resulta vacía.


con los datos actuales el modelo general es el único util y cumple entregando los resultados.